# 01 — Stage 1: SCLIP-style region extraction

Runs the frozen SigLIP-B/16 vision tower with correlation attention in its
final block, clusters the resulting patch features into `num_regions`
groups, and saves the bounding-box crops + metadata under
`cfg.data.regions_out`.

In [ ]:
%cd /content/region-grounded
import json
import torch
from region_grounded import load_config
from region_grounded.data import CC3MIndex
from region_grounded.stage1_extract import SCLIPVisionEncoder, extract_and_save

cfg = load_config('configs/default.yaml')
index = CC3MIndex(cfg.data.cc3m_root, f'{cfg.data.cc3m_root}/index.jsonl', limit=cfg.data.subset_size)
print(len(index), 'global samples')

In [ ]:
encoder = SCLIPVisionEncoder(cfg.stage1.siglip_model, dtype=torch.float16)
pairs = [(s.image_path, s.caption) for s in index]
records = extract_and_save(pairs, encoder, cfg.stage1, out_dir=cfg.data.regions_out, seed=cfg.data.seed)
with open('outputs/stage1_records.jsonl', 'w') as f:
    for r in records:
        f.write(json.dumps(r) + '\n')
print('wrote', len(records), 'records')

In [ ]:
# Sanity viz on the first sample
import matplotlib.pyplot as plt
from PIL import Image
r = records[0]
fig, axes = plt.subplots(1, 1 + len(r['regions']), figsize=(3 * (1 + len(r['regions'])), 3))
axes[0].imshow(Image.open(r['image_path'])); axes[0].set_title(r['global_caption'][:40]); axes[0].axis('off')
for i, rg in enumerate(r['regions']):
    axes[i + 1].imshow(Image.open(rg['region_path'])); axes[i + 1].axis('off')
plt.show()